# 14 — Gemma generation benchmark: prompt → Tunix generation → decode/env step

Отдельная тетрадка для замеров generation pipeline. Она не тренирует модель: только прогоняет несколько batched rollout runs и сохраняет JSON с latency, env-steps/sec, generated tokens/sec и phase profile.

Требуется локальный snapshot `artifacts/models/gemma3-270m-it`; автоскачивания нет.


In [ ]:
from pathlib import Path
import json
import time

import jax
import jax.numpy as jnp

from tunix.models.automodel import call_model_config

from tunix_craftext.rollouts.batched import collect_batched_text_rollout
from tunix_craftext.env.config import load_mvp_config
from tunix_craftext.artifacts.profiling import PhaseProfiler, save_profile
from tunix_craftext.env.prompts import MegaPromptRenderer
from tunix_craftext.env.runtime import build_craftext_runtime
from tunix_craftext.env.tasks import CrafTextTaskSampler
from tunix_craftext.models.tunix_actor import build_gemma_tunix_actor, init_linear_value_head


In [ ]:
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the tunix-craftext repository')

CONFIG_PATH = ROOT / 'configs/env/smoke/tiny_craftext.yaml'
MODEL_PROFILE = ROOT / 'configs/models/gemma3_270m_instruction.yaml'
SNAPSHOT = ROOT / 'artifacts/models/gemma3-270m-it'
BENCHMARK_PATH = ROOT / 'artifacts/benchmarks/gemma-generation-notebook.json'
PROFILE_PATH = ROOT / 'artifacts/profiles/gemma-generation-notebook.json'

BATCH_SIZES = (1, 2, 4)
HORIZONS = (1, 2, 4)
REPEATS = 3
MAX_NEW_TOKENS = 12
CACHE_SIZE = 1024
SEED = 23

if not SNAPSHOT.is_dir():
    raise FileNotFoundError(f'Local Gemma snapshot is required: {SNAPSHOT}')


In [ ]:
config = load_mvp_config(CONFIG_PATH)
runtime = build_craftext_runtime(config)
task_sampler = CrafTextTaskSampler.from_runtime(
    runtime,
    horizon=config.environment.horizon,
    mode='fixed',
    fixed_instruction_index=config.environment.instruction_index,
)
prepared_goal, prepared_instruction_index = task_sampler.task_at(config.environment.instruction_index)
renderer = MegaPromptRenderer(config.prompt.template)
gemma_config = call_model_config('gemma3_270m_it')
actor = build_gemma_tunix_actor(
    profile_path=MODEL_PROFILE,
    snapshot=SNAPSHOT,
    cache_size=CACHE_SIZE,
    value_head=init_linear_value_head(jax.random.PRNGKey(SEED), hidden_dim=gemma_config.embed_dim),
    seed=SEED,
)
print('model:', actor.profile.model_id, 'actions:', runtime.action_count)
print('prepared CrafText task:', prepared_instruction_index, prepared_goal)


In [ ]:
profiler = PhaseProfiler()
rows = []

for batch_size in BATCH_SIZES:
    for horizon in HORIZONS:
        for repeat in range(REPEATS):
            started = time.perf_counter()
            with profiler.section('rollout'):
                rollout = collect_batched_text_rollout(
                    runtime.adapter,
                    renderer,
                    actor.backend,
                    actions=runtime.actions,
                    batch_size=batch_size,
                    horizon=horizon,
                    seed=SEED + repeat + 100 * batch_size + 1000 * horizon,
                    goal=prepared_goal,
                    max_new_tokens=MAX_NEW_TOKENS,
                    invalid_action='fallback',
                    fallback_action_id=0,
                )
            elapsed = time.perf_counter() - started
            tokens = sum(
                len(response.token_ids or ())
                for decision in rollout.decisions
                for response in decision.responses
            )
            fallback_count = sum(
                int(jnp.sum(decision.fallback_used)) for decision in rollout.decisions
            )
            env_steps = batch_size * horizon
            rows.append({
                'batch_size': batch_size,
                'horizon': horizon,
                'repeat': repeat,
                'elapsed_seconds': elapsed,
                'env_steps': env_steps,
                'env_steps_per_second': env_steps / elapsed if elapsed else 0.0,
                'generated_tokens': tokens,
                'generated_tokens_per_second': tokens / elapsed if elapsed else 0.0,
                'fallback_count': fallback_count,
            })

BENCHMARK_PATH.parent.mkdir(parents=True, exist_ok=True)
BENCHMARK_PATH.write_text(json.dumps({
    'schema': 'tunix-craftext.generation-benchmark/v1',
    'model': actor.profile.model_id,
    'batch_sizes': BATCH_SIZES,
    'horizons': HORIZONS,
    'repeats': REPEATS,
    'rows': rows,
}, indent=2, sort_keys=True), encoding='utf-8')
save_profile(PROFILE_PATH, profiler.events(), metadata={'notebook': '14_generation_benchmark.ipynb'})
print('benchmark saved:', BENCHMARK_PATH)
print('profile saved:', PROFILE_PATH)
rows[:3]


In [ ]:
# Compact table grouped by batch/horizon.
from collections import defaultdict

groups = defaultdict(list)
for row in rows:
    groups[(row['batch_size'], row['horizon'])].append(row)

for (batch_size, horizon), values in sorted(groups.items()):
    mean_steps = sum(row['env_steps_per_second'] for row in values) / len(values)
    mean_tokens = sum(row['generated_tokens_per_second'] for row in values) / len(values)
    mean_fallback = sum(row['fallback_count'] for row in values) / len(values)
    print(
        f'B={batch_size:>2} H={horizon:>2}: '
        f'{mean_steps:8.2f} env steps/s, '
        f'{mean_tokens:8.2f} gen tokens/s, '
        f'fallback avg={mean_fallback:.2f}'
    )
